# Patches — Bloc 5b, Sessió 9
### Arquitectura UGen, control rate (vibrato), FM synthesis (timbre)

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Per guardar els teus canvis: `File → Save a copy in Drive`.

⚠️ **Important:** aquest notebook cobreix la part **conceptual, vibrato i FM** de la Sessió 9 (Seccions 1-4 de `exemples.py`). La integració amb **MIDI en temps real** (Secció 6) NOMÉS funciona a **Thonny** — Colab no té accés al hardware d'àudio del teu ordinador. Aquí escoltarem el so renderitzat amb `Audio()`, no en temps real.

In [ ]:
import numpy as np
from IPython.display import Audio, display
import matplotlib.pyplot as plt

sample_rate = 44100

## 1. El problema: `generate(duration)` no recorda la fase (recordatori S8)

In [ ]:
class OscillatorBatch:
    """Versió de la Sessió 8 — recordatori."""
    def __init__(self, freq=440.0, sample_rate=44100):
        self.freq = freq
        self.sample_rate = sample_rate

    def generate(self, duration):
        n = int(self.sample_rate * duration)
        t = np.linspace(0, duration, n, endpoint=False)
        return np.sin(2 * np.pi * self.freq * t)

osc_batch = OscillatorBatch(freq=440)
bloc1 = osc_batch.generate(0.1)
bloc2 = osc_batch.generate(0.1)  # torna a començar a t=0!
junts = np.concatenate([bloc1, bloc2])

print("Escolta: hauries de notar un petit click entre els dos blocs.")
display(Audio(junts, rate=sample_rate))

In [ ]:
# Visualitzem el "salt" de fase a la unió dels dos blocs
plt.figure(figsize=(10, 3))
plt.plot(junts[:300])
plt.axvline(x=len(bloc1), color='r', linestyle='--', label='unió bloc1/bloc2')
plt.title("El salt de fase es veu com una discontinuïtat aquí")
plt.legend()
plt.show()

## 2. L'arquetip UGen: `process(n_samples)` amb estat persistent

Connexió amb Max/Pd: aquest patró —rebre/enviar blocs de senyal mantenint estat propi— és exactament el model d'un patch modular.

In [ ]:
class Oscillator:
    def __init__(self, freq=440.0, waveform='sine', sample_rate=44100):
        self.freq = freq
        self.waveform = waveform
        self.sample_rate = sample_rate
        self.phase = 0.0  # ESTAT: fase acumulada

    def process(self, n_samples):
        phase_inc = 2 * np.pi * self.freq / self.sample_rate
        phases = self.phase + phase_inc * np.arange(n_samples)

        if self.waveform == 'sine':
            out = np.sin(phases)
        elif self.waveform == 'square':
            out = np.sign(np.sin(phases))
        elif self.waveform == 'sawtooth':
            out = 2 * ((phases / (2 * np.pi)) % 1.0) - 1.0
        else:
            raise ValueError(f"Forma d'ona desconeguda: {self.waveform}")

        self.phase = (phases[-1] + phase_inc) % (2 * np.pi)
        return out

osc = Oscillator(freq=440)
bloc1 = osc.process(4410)
bloc2 = osc.process(4410)
junts = np.concatenate([bloc1, bloc2])

print("Ara hauria de sonar continu, sense click:")
display(Audio(junts, rate=sample_rate))

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(junts[:300])
plt.axvline(x=len(bloc1), color='g', linestyle='--', label='unió bloc1/bloc2 (ara sense salt)')
plt.title("Amb process(), la fase és contínua")
plt.legend()
plt.show()

## 3. Control rate vs. audio rate: vibrato

**Control rate de debò (en un motor DSP real):** un generador de control (LFO, envolupant) calcula els seus propis valors a una freqüència molt inferior a la d'àudio — p.ex. 100-200 valors/segon en lloc de 44100/s — perquè el paràmetre que controla no necessita més resolució.

**⚠️ El nostre `LFO` és un HÍBRID PEDAGÒGIC, no control rate real:** per simplicitat, reutilitza exactament la mateixa lògica que `Oscillator` — calcula a 44100Hz igual que el portador, NO genera realment menys valors. "Fingim" que està a control rate llegint-ne només l'última mostra de cada bloc.

In [ ]:
class LFO:
    """HÍBRID: mateixa lògica que Oscillator (calcula a 44100Hz). NO és
    un generador de control rate real — només en llegim l'última mostra
    del bloc, com si ho fos."""
    def __init__(self, freq=5.0, sample_rate=44100):
        self.freq = freq
        self.sample_rate = sample_rate
        self.phase = 0.0

    def process(self, n_samples):
        phase_inc = 2 * np.pi * self.freq / self.sample_rate
        phases = self.phase + phase_inc * np.arange(n_samples)
        out = np.sin(phases)
        self.phase = (phases[-1] + phase_inc) % (2 * np.pi)
        return out


def vibrato(base_freq, lfo_freq, depth, duration, sample_rate=44100, block=512):
    n_total = int(sample_rate * duration)
    sortida = np.zeros(n_total, dtype='float32')

    lfo = LFO(freq=lfo_freq)
    osc = Oscillator(freq=base_freq, waveform='sine')

    i = 0
    while i < n_total:
        n = min(block, n_total - i)
        lfo_out = lfo.process(n)
        osc.freq = base_freq + depth * lfo_out[-1]   # llegim el LFO a "control rate": 1 valor/bloc
        sortida[i:i+n] = osc.process(n)
        i += n

    return sortida

print("Nota de 440Hz SENSE vibrato:")
sense_vibrato = vibrato(base_freq=440, lfo_freq=5.0, depth=0.0, duration=1.5)
display(Audio(sense_vibrato, rate=sample_rate))

In [ ]:
print("Mateixa nota AMB vibrato (LFO 5Hz, excursió de només 6Hz):")
amb_vibrato = vibrato(base_freq=440, lfo_freq=5.0, depth=6.0, duration=1.5)
display(Audio(amb_vibrato, rate=sample_rate))
print("(hauries de sentir la MATEIXA nota, ondulant suaument — no un timbre nou)")
print()
print("Per què funciona igualment, tot i ser un híbrid: el LFO es mou lent (5Hz),")
print("així que el seu valor amb prou feines canvia dins un bloc — llegir-ne")
print("només l'última mostra no perd informació rellevant. NO és un estalvi de")
print("CPU: process() calcula el bloc sencer igualment.")

## 4. FM synthesis: UGens encadenats, però a AUDIO RATE

Diferència de fons amb el vibrato: la FM és, per definició, modulació a audio rate — el modulador es mou dins el rang audible. Si el llegim només 1 cop/bloc (com fem aquí per simplicitat), estem sota-mostrejant-lo: introduïm artefactes/aliasing.

`I` = índex de modulació (notació estàndard: Chowning, DX7).

In [ ]:
def fm_synth(carrier_freq, mod_freq, I, duration, sample_rate=44100, block=512):
    n_total = int(sample_rate * duration)
    sortida = np.zeros(n_total, dtype='float32')

    modulator = Oscillator(freq=mod_freq, waveform='sine')
    carrier = Oscillator(freq=carrier_freq, waveform='sine')

    i = 0
    while i < n_total:
        n = min(block, n_total - i)
        mod_out = modulator.process(n)
        carrier.freq = carrier_freq + I * mod_out[-1]
        sortida[i:i+n] = carrier.process(n)
        i += n

    return sortida

so_fm = fm_synth(carrier_freq=440, mod_freq=80, I=200, duration=2.0)
print("FM: portador 440Hz, modulador 80Hz (audio rate), I=200Hz")
print("Compara amb el vibrato d'abans: ja no reconeixes 'una nota ondulant',")
print("sona com un timbre diferent. (Simplificació amb artefactes: carrier.freq")
print("s'actualitza només 1 cop/bloc — veure Challenge per la versió correcta.)")
display(Audio(so_fm, rate=sample_rate))

In [ ]:
so_fm_alt = fm_synth(carrier_freq=440, mod_freq=80, I=500, duration=2.0)
print("Mateixa FM amb I=500Hz (encara més 'metàl·lic'):")
display(Audio(so_fm_alt, rate=sample_rate))

In [ ]:
# Comparativa espectral objectiva: vibrato (control) vs FM (timbre)
def amplada_de_banda(signal, sample_rate=44100, threshold_db=-20):
    spec = np.abs(np.fft.rfft(signal))
    spec_db = 20 * np.log10(spec / np.max(spec) + 1e-12)
    freqs = np.fft.rfftfreq(len(signal), 1 / sample_rate)
    mask = spec_db > threshold_db
    return freqs[mask].max() - freqs[mask].min() if np.any(mask) else 0

bw_vibrato = amplada_de_banda(amb_vibrato)
bw_fm = amplada_de_banda(so_fm)

print(f"Amplada de banda vibrato (depth=6):  {bw_vibrato:.0f} Hz")
print(f"Amplada de banda FM (I=200):          {bw_fm:.0f} Hz")
print(f"\nLa FM ocupa {bw_fm/bw_vibrato:.0f}x més espectre — per això es percep")
print("com un timbre nou i no com una nota que ondula.")

fig, axs = plt.subplots(1, 2, figsize=(12, 3))
for ax, sig, title in [(axs[0], amb_vibrato, "Vibrato (control rate: modulador lent)"),
                         (axs[1], so_fm, "FM (timbre)")]:
    spec = np.abs(np.fft.rfft(sig[:8192]))
    freqs = np.fft.rfftfreq(8192, 1/sample_rate)
    ax.plot(freqs[:400], spec[:400])
    ax.set_title(title)
    ax.set_xlabel("Hz")
plt.tight_layout()
plt.show()

## 5. Per provar tu mateix

Canvia `depth` a `vibrato()` i `I` a `fm_synth()` i escolta on és la frontera entre "encara sona a vibrato" i "ja sona a timbre nou".

In [ ]:
# El teu espai per experimentar
so_experiment = fm_synth(carrier_freq=300, mod_freq=150, I=350, duration=2.0)
display(Audio(so_experiment, rate=sample_rate))

## Recordatori

- La integració amb **MIDI en temps real** (callback de `sounddevice` + `mido`) es treballa a **Thonny**, no aquí — vegeu `exemples.py` Secció 6/6b.
- El mini-repte d'aquesta sessió (`assignment.py`, amb `vibrato()` i `fm_synth()`) també és per a **Thonny**.
- **Sessió 10:** consolidació de tot el Bloc 5 (Oscillator + Envelope + UGen + vibrato + FM + MIDI temps real).